In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from shadowroutes.config import settings

src_path = Path('../')
os.chdir(src_path)

## Parameters

In [17]:
# Safety and institutional performance perception (ENVIPE 2025 - INEGI) - obtained in November 2025
envipe_path = Path(settings.DATA_ROOT / 'envipe-inegi' / 'raw' / 'conjunto_de_datos_envipe_2025_csv' / 'tper_vic1_envipe2025' / 'conjunto_de_datos' / 'conjunto_de_datos_tper_vic1_envipe2025.csv')
dict_envipe_path = Path(settings.DATA_ROOT / 'envipe-inegi' / 'raw' / 'conjunto_de_datos_envipe_2025_csv' / 'tper_vic1_envipe2025' / 'diccionario_de_datos' / 'diccionario_de_datos_tper_vic1_envipe2025.csv')

# Population estimates (mid-year) from CONAPO - obtained in October 2025
data_path_pop_m = Path(settings.DATA_ROOT / 'conapo' / 'raw' / 'pobproy_quinq1.csv')

# Output path
out_collusion_panel = Path(settings.DATA_ROOT / 'processed' / 'collusion_index_panel.csv')


## Data

In [3]:
# Loads data related to personnel in police corporations
df_envipe = pd.read_csv(envipe_path, encoding='utf-8', dtype={"CVE_ENT": str, "CVE_MUN": str})
print(f'File read: {print(envipe_path.name)}')

print(df_envipe.shape)
df_envipe.head()

conjunto_de_datos_tper_vic1_envipe2025.csv
File read: None
(91182, 240)


,ID_VIV,ID_HOG,ID_PER,UPM,VIV_SEL,HOGAR,RESUL_H,R_SEL,SEXO,EDAD,...,AP5_8_11_5,AP5_9,FAC_HOG,FAC_ELE,FAC_HOG_AM,FAC_ELE_AM,DOMINIO,ESTRATO,EST_DIS,UPM_DIS
0,100107.01,0100107.01.01,0100107.01.01.01,100107,1,1,B,1,2,56,...,NaN,3,241,241,241.0,241.0,U,2,1,47
1,100107.02,0100107.02.01,0100107.02.01.02,100107,2,1,A,2,2,30,...,9.0,2,241,482,241.0,482.0,U,2,1,47
2,100107.03,0100107.03.01,0100107.03.01.04,100107,3,1,B,4,2,18,...,NaN,9,241,963,241.0,963.0,U,2,1,47
3,100107.04,0100107.04.01,0100107.04.01.03,100107,4,1,A,3,1,22,...,9.0,2,241,723,241.0,723.0,U,2,1,47
4,100107.05,0100107.05.01,0100107.05.01.01,100107,5,1,B,1,2,36,...,NaN,2,241,482,241.0,482.0,U,2,1,47


In [4]:
df_envipe.CVE_MUN.unique()

array(['001', '002', '003', '005', '006', '007', '010', '011', '004',
       '008', '009', '012', '017', '018', '020', '022', '024', '025',
       '027', '028', '030', '032', '033', '035', '036', '014', '015',
       '019', '023', '040', '043', '047', '048', '052', '059', '061',
       '065', '068', '071', '074', '078', '079', '083', '087', '089',
       '094', '096', '097', '099', '101', '102', '106', '108', '110',
       '111', '112', '124', '081', '016', '021', '031', '037', '041',
       '044', '051', '062', '064', '067', '069', '072', '077', '084',
       '090', '093', '100', '109', '114', '115', '045', '050', '055',
       '013', '026', '029', '038', '039', '042', '046', '053', '056',
       '057', '066', '075', '058', '063', '076', '082', '049', '054',
       '073', '080', '070', '085', '098', '103', '120', '034', '125',
       '060', '091', '092', '095', '104', '118', '121', '122', '086',
       '088', '107', '126', '143', '150', '157', '166', '177', '184',
       '190', '197',

In [5]:
# Loads data related to personnel in police corporations
df_envipe_dict = pd.read_csv(dict_envipe_path, encoding='utf-8')
print(f'File read: {print(dict_envipe_path.name)}')

print(df_envipe_dict.shape)
df_envipe_dict.head()

diccionario_de_datos_tper_vic1_envipe2025.csv
File read: None
(903, 5)


,NOMBRE_CAMPO,NEMONICO,TIPO,LONGITUD,RANGO_CLAVES
0,Identificador de vivienda seleccionada,ID_VIV,Alfanumérico,10,"0100001.01,…,3299999.99"
1,Identificador del hogar seleccionado,ID_HOG,Alfanumérico,13,"0100001.01.01,…,3299999.99.99"
2,Identificador del informante seleccionado,ID_PER,Alfanumérico,16,"0100001.01.01.01,…,3299999.99.99.30"
3,Control de vivienda (UPM),UPM,Carácter,7,"0100001,…,3299999"
4,Vivienda seleccionada,VIV_SEL,Carácter,2,"01,…,99"


In [6]:
dict_vars = dict(zip(df_envipe_dict["NEMONICO"], df_envipe_dict["NOMBRE_CAMPO"], strict=False))
dict_vars

{'ID_VIV': 'Identificador de vivienda seleccionada',
 'ID_HOG': 'Identificador del hogar seleccionado',
 'ID_PER': 'Identificador del informante seleccionado',
 'UPM': 'Control de vivienda (UPM)',
 'VIV_SEL': 'Vivienda seleccionada',
 'HOGAR': 'Control del hogar',
 'RESUL_H': 'Resultado de la visita al hogar',
 'R_SEL': 'Número de renglón de la persona seleccionada',
 'SEXO': 'Sexo',
 'EDAD': 'Edad',
 'AREAM': 'Área Urbana de Interés',
 'CVE_ENT': 'Clave Entidad',
 'NOM_ENT': 'Nombre de la Entidad',
 'CVE_MUN': 'Municipio',
 'NOM_MUN': 'Nombre del municipio',
 'AP4_1': 'Tiempo habitando en la vivienda',
 'AP4_2_01': 'Temas que más preocupan: pobreza',
 'AP4_2_02': 'Temas que más preocupan: desempleo',
 'AP4_2_03': 'Temas que más preocupan: narcotráfico',
 'AP4_2_04': 'Temas que más preocupan: aumento de precios',
 'AP4_2_05': 'Temas que más preocupan: inseguridad',
 'AP4_2_06': 'Temas que más preocupan: desastres naturales',
 'AP4_2_07': 'Temas que más preocupan: escasez de agua',
 'AP

### Preprocessing

In [7]:
df_envipe["muni_id"] = df_envipe["CVE_ENT"] + df_envipe["CVE_MUN"]
df_envipe.head()

,ID_VIV,ID_HOG,ID_PER,UPM,VIV_SEL,HOGAR,RESUL_H,R_SEL,SEXO,EDAD,...,AP5_9,FAC_HOG,FAC_ELE,FAC_HOG_AM,FAC_ELE_AM,DOMINIO,ESTRATO,EST_DIS,UPM_DIS,muni_id
0,100107.01,0100107.01.01,0100107.01.01.01,100107,1,1,B,1,2,56,...,3,241,241,241.0,241.0,U,2,1,47,01001
1,100107.02,0100107.02.01,0100107.02.01.02,100107,2,1,A,2,2,30,...,2,241,482,241.0,482.0,U,2,1,47,01001
2,100107.03,0100107.03.01,0100107.03.01.04,100107,3,1,B,4,2,18,...,9,241,963,241.0,963.0,U,2,1,47,01001
3,100107.04,0100107.04.01,0100107.04.01.03,100107,4,1,A,3,1,22,...,2,241,723,241.0,723.0,U,2,1,47,01001
4,100107.05,0100107.05.01,0100107.05.01.01,100107,5,1,B,1,2,36,...,2,241,482,241.0,482.0,U,2,1,47,01001


In [8]:
df_envipe = df_envipe[df_envipe.CVE_ENT.isin(['12', '16', '15'])]
df_envipe.NOM_ENT.unique()

array(['GUERRERO', 'MEXICO', 'MICHOACAN DE OCAMPO'], dtype=object)

### Collusion index

**a) Perception of security & local crime environment**

* AP4_3_2 – Percepción sobre seguridad en su municipio o demarcación territorial
* AP4_5_15 – Incivilidades en su colonia: extorsiones (cobro de piso)
* AP4_5_16 – Incivilidades en su colonia: robo o venta ilegal de gasolina o diésel (huachicol)

**b) Trust & perceived corruption in authorities**

Municipal police:
* AP5_4_02 – Confianza en Policía Preventiva Municipal
* AP5_5_02 – Percepción sobre corrupción de Policía Preventiva Municipal
* AP5_6_02 – Percepción sobre desempeño de Policía Preventiva Municipal

State police:
* AP5_4_03 – Confianza en Policía Estatal
* AP5_5_03 – Percepción sobre corrupción de Policía Estatal
* AP5_6_03 – Percepción sobre desempeño de Policía Estatal

In [9]:
id_cols = ['CVE_ENT', 'NOM_ENT', 'muni_id', 'NOM_MUN']
weights = ['FAC_PER']  # personal-level weight
sec_per = ['AP4_3_2']
trust_per = ['AP5_4_02', 'AP5_4_03']
corr_per = ['AP5_5_02', 'AP5_5_03']
perf_per = ['AP5_6_02', 'AP5_6_03']

In [10]:
# 1) Insecurity in municipality (AP4_3_2)
sec_map = {1: 0.0,  # seguro
           2: 1.0}  # inseguro

df_envipe["insecurity_mun"] = df_envipe["AP4_3_2"].map(sec_map)

# 2) Trust in municipal & state police
trust_map = {
    1: 1.0,   # mucha confianza
    2: 2/3,   # algo de confianza
    3: 1/3,   # algo de desconfianza
    4: 0.0    # mucha desconfianza
}

for col in trust_per:
    df_envipe[f"trust_{col}"] = df_envipe[col].map(trust_map)

# Average trust across municipality + state police
df_envipe["trust_police"] = df_envipe[[f"trust_{c}" for c in trust_per]].mean(axis=1)

# Turn into mistrust (0=trust, 1=mistrust)
df_envipe["mistrust_police"] = 1 - df_envipe["trust_police"]

# 3) Perceived corruption (yes/no)
corr_map = {
    1: 1.0,  # Sí son corruptos
    2: 0.0   # No
}

for col in corr_per:
    df_envipe[f"corr_{col}"] = df_envipe[col].map(corr_map)

df_envipe["corr_police"] = df_envipe[[f"corr_{c}" for c in corr_per]].mean(axis=1)

# 4) Performance → ineffectiveness
ineff_map = {
    1: 0.0,  # muy efectivo
    2: 1/3,
    3: 2/3,
    4: 1.0  # nada efectivo
}

for col in perf_per:
    df_envipe[f"ineff_{col}"] = df_envipe[col].map(ineff_map)

df_envipe["ineff_police"] = df_envipe[[f"ineff_{c}" for c in perf_per]].mean(axis=1)

In [11]:
df_envipe.head()

,ID_VIV,ID_HOG,ID_PER,UPM,VIV_SEL,HOGAR,RESUL_H,R_SEL,SEXO,EDAD,...,trust_AP5_4_02,trust_AP5_4_03,trust_police,mistrust_police,corr_AP5_5_02,corr_AP5_5_03,corr_police,ineff_AP5_6_02,ineff_AP5_6_03,ineff_police
33100,1200174.03,1200174.03.01,1200174.03.01.01,1200174,3,1,B,1,2,67,...,0.666667,0.666667,0.666667,0.333333,1.0,1.0,1.0,0.666667,0.666667,0.666667
33101,1200174.04,1200174.04.01,1200174.04.01.04,1200174,4,1,A,4,1,50,...,0.666667,0.666667,0.666667,0.333333,1.0,1.0,1.0,0.333333,0.333333,0.333333
33102,1200177.01,1200177.01.01,1200177.01.01.01,1200177,1,1,B,1,2,40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33103,1200177.02,1200177.02.01,1200177.02.01.02,1200177,2,1,A,2,1,38,...,0.000000,0.000000,0.000000,1.000000,1.0,1.0,1.0,0.666667,0.666667,0.666667
33104,1200177.03,1200177.03.01,1200177.03.01.01,1200177,3,1,B,1,2,77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
def wavg(series, weights):
    mask = (~series.isna()) & (~weights.isna())
    if mask.sum() == 0:
        return np.nan
    return np.average(series[mask], weights=weights[mask])

envipe_mun = (
    df_envipe.groupby(id_cols)
      .apply(lambda g: pd.Series({
          "insecurity_mun":  wavg(g["insecurity_mun"],  g["FAC_ELE"]),
          "mistrust_police": wavg(g["mistrust_police"], g["FAC_ELE"]),
          "corr_police":     wavg(g["corr_police"],     g["FAC_ELE"]),
          "ineff_police":    wavg(g["ineff_police"],    g["FAC_ELE"]),
      }))
      .reset_index()
)

C:\Users\penny\AppData\Local\Temp\ipykernel_8728\1095217559.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [13]:
s = (
    0.35 * envipe_mun["corr_police"] +
    0.25 * envipe_mun["mistrust_police"] +
    0.20 * envipe_mun["ineff_police"] +
    0.20 * envipe_mun["insecurity_mun"]
)

envipe_mun["collusion_idx_raw"] = s
envipe_mun["collusion_idx"] = ((s - s.min()) / (s.max() - s.min()))


In [14]:
envipe_mun.head()

,CVE_ENT,NOM_ENT,muni_id,NOM_MUN,insecurity_mun,mistrust_police,corr_police,ineff_police,collusion_idx_raw,collusion_idx
0,12,GUERRERO,12001,ACAPULCO DE JUAREZ,0.891895,0.614207,0.773391,0.604499,0.723517,0.783212
1,12,GUERRERO,12003,AJUCHITLAN DEL PROGRESO,0.558824,0.333333,0.769231,0.566667,0.577662,0.572155
2,12,GUERRERO,12006,APAXTLA,0.272689,0.298229,0.708343,0.429791,0.462973,0.406196
3,12,GUERRERO,12007,ARCELIA,0.332788,0.380682,0.426025,0.382584,0.387353,0.296772
4,12,GUERRERO,12010,ATLIXTAC,0.500039,0.484122,0.342120,0.460336,0.432848,0.362603


In [15]:
envipe_mun.columns = envipe_mun.columns.str.lower()
envipe_mun.rename(columns={'cve_ent': 'state_id', 'nom_ent': 'state',  
                          'nom_mun': 'municipality'}, inplace=True)

envipe_mun['year'] = 2025
envipe_mun['year'] = envipe_mun['year'].astype('int64')
envipe_mun['state_id'] = envipe_mun['state_id'].astype('int64')
envipe_mun['state'] = envipe_mun['state'].astype('str')
envipe_mun['muni_id'] = envipe_mun['muni_id'].astype('str')
envipe_mun['municipality'] = envipe_mun['municipality'].astype('str')


In [16]:
print(envipe_mun.shape)
envipe_mun.head(10)

(163, 11)


,state_id,state,muni_id,municipality,insecurity_mun,mistrust_police,corr_police,ineff_police,collusion_idx_raw,collusion_idx,year
0,12,GUERRERO,12001,ACAPULCO DE JUAREZ,0.891895,0.614207,0.773391,0.604499,0.723517,0.783212,2025
1,12,GUERRERO,12003,AJUCHITLAN DEL PROGRESO,0.558824,0.333333,0.769231,0.566667,0.577662,0.572155,2025
2,12,GUERRERO,12006,APAXTLA,0.272689,0.298229,0.708343,0.429791,0.462973,0.406196,2025
3,12,GUERRERO,12007,ARCELIA,0.332788,0.380682,0.426025,0.382584,0.387353,0.296772,2025
4,12,GUERRERO,12010,ATLIXTAC,0.500039,0.484122,0.342120,0.460336,0.432848,0.362603,2025
5,12,GUERRERO,12011,ATOYAC DE ALVAREZ,0.427127,0.365157,0.503886,0.402423,0.433559,0.363633,2025
6,12,GUERRERO,12012,AYUTLA DE LOS LIBRES,0.818782,0.458868,0.444611,0.485167,0.531121,0.504807,2025
7,12,GUERRERO,12019,COPALILLO,0.611446,0.443592,0.518358,0.494558,0.513524,0.479345,2025
8,12,GUERRERO,12020,COPANATOYAC,0.625429,0.500244,0.533988,0.566435,0.550330,0.532603,2025
9,12,GUERRERO,12021,COYUCA DE BENITEZ,0.688688,0.490717,0.520528,0.462088,0.535019,0.510449,2025


#### Save data

In [18]:
envipe_mun.to_csv(out_collusion_panel, index=False)